In [ ]:
import pandas as pd
from pathlib import Path
import numpy as np
import seaborn as sns
from scipy import stats

from itertools import combinations
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests

In [ ]:
sns.set_context("paper", font_scale=1.4)

colors = [
    "#394C89", "#5068AF","#93A4DC", "#7A7A7A", "#BFBFBF", "#9D7195", "#C4A9BA", "#88B9B5",
]

In [ ]:
base_path = Path("/path/to/ICD11_WHO")
results_folder = base_path.joinpath("results_final","statistics")
results_folder.mkdir(exist_ok=True)

## Tests to check Model's performance is non-random

In [ ]:
alpha = 0.05

# -------------------------------------------------------------------------
# --- FUNCTION: top-n significance testing
# -------------------------------------------------------------------------
def significance_tests_top_n(data, n_diagnoses, model_name, category):
    """
    Run Top-n significance tests vs. random baseline and return a results DataFrame.
    Includes CI and 'correct' counts.
    """
    N = len(data)
    results = []

    # Observed accuracies
    accs = {k: data[f'Top_{k}_Accuracy'].mean() for k in [1, 2, 3]}
    counts = {k: data[f'Top_{k}_Accuracy'].sum() for k in [1, 2, 3]}
    expected = {k: k / n_diagnoses for k in [1, 2, 3]}

    # --- Top-k tests ---
    for k in [1, 2, 3]:
        count = counts[k]
        stat, pval = proportions_ztest(count, N, value=expected[k], alternative='larger')
        ci_low, ci_upp = proportion_confint(count, N, alpha=alpha, method='wilson')

        results.append({
            "Model": model_name,
            "Category": category,
            "Test_Type": f"Top-{k}",
            "Observed_Accuracy": accs[k],
            "Correct_Count": int(count),
            "N": N,
            "Expected_Random": expected[k],
            "z_value": stat,
            "p_value": pval,
            "CI_Lower": ci_low,
            "CI_Upper": ci_upp,
            "n_diagnoses": n_diagnoses
        })

    # --- Improvement tests (Top-1→Top-2, Top-2→Top-3) ---
    improvement_pairs = [(1, 2), (2, 3)]
    for (a, b) in improvement_pairs:
        improvement = (data[f'Top_{b}_Accuracy'] & ~data[f'Top_{a}_Accuracy'])
        num_improved = improvement.sum()
        p_improvement = num_improved / N
        p_expected = (b - a) / n_diagnoses
        stat, pval = proportions_ztest(num_improved, N, value=p_expected, alternative='larger')
        ci_low, ci_upp = proportion_confint(num_improved, N, alpha=alpha, method='wilson')

        results.append({
            "Model": model_name,
            "Category": category,
            "Test_Type": f"Improvement Top{a}→Top{b}",
            "Observed_Accuracy": p_improvement,
            "Correct_Count": int(num_improved),
            "N": N,
            "Expected_Random": p_expected,
            "z_value": stat,
            "p_value": pval,
            "CI_Lower": ci_low,
            "CI_Upper": ci_upp,
            "n_diagnoses": n_diagnoses
        })

    return pd.DataFrame(results)


# -------------------------------------------------------------------------
# --- FUNCTION: apply multiple correction
# -------------------------------------------------------------------------
def apply_multiple_corrections(df, alpha=0.05):
    """Apply Holm and FDR corrections to all p-values."""
    df = df.copy()
    valid = df['p_value'].notna()
    if valid.sum() == 0:
        return df

    df.loc[valid, 'p_holm'] = multipletests(df.loc[valid, 'p_value'], method='holm')[1]
    df.loc[valid, 'p_fdr'] = multipletests(df.loc[valid, 'p_value'], method='fdr_bh')[1]

    df['significant_holm'] = df['p_holm'] < alpha
    df['significant_fdr'] = df['p_fdr'] < alpha
    df['Correction_Significant'] = np.select(
        [df['significant_holm'], df['significant_fdr']],
        ['Holm', 'FDR'],
        default='None'
    )
    return df


# -------------------------------------------------------------------------
# --- LOOP OVER MODELS AND CATEGORIES
# -------------------------------------------------------------------------
n_choices = {"Mood": 13, "Stress": 8, "Anxiety": 16}
all_results = []

files = sorted(results_folder.parent.glob("*/english/*detailed_results_corrected.csv"))

for file in files:
    model = file.parts[-3]
    df = pd.read_csv(file, index_col=['Vignette_ID', 'Category'])

    # --- Category-level ---
    for category in ["Anxiety", "Mood", "Stress"]:
        if category not in df.index.get_level_values("Category"):
            continue
        n_diagnoses = n_choices[category]
        df_cat = df.xs(category, level="Category")[['Top_1_Accuracy', 'Top_2_Accuracy', 'Top_3_Accuracy']]
        res_df = significance_tests_top_n(df_cat, n_diagnoses, model, category)
        all_results.append(res_df)

    # --- Overall (all vignettes pooled) ---
    N_all = len(df)
    n_diagnoses_overall = int(np.mean(list(n_choices.values())))  # or use len(unique ICDs) if known
    res_overall = significance_tests_top_n(
        df[['Top_1_Accuracy', 'Top_2_Accuracy', 'Top_3_Accuracy']],
        n_diagnoses_overall,
        model,
        category="Overall"
    )
    all_results.append(res_overall)


# -------------------------------------------------------------------------
# --- COMBINE + CORRECT
# -------------------------------------------------------------------------
final_results = pd.concat(all_results, ignore_index=True)
final_results = apply_multiple_corrections(final_results, alpha=alpha)

# -------------------------------------------------------------------------
# --- SAVE
# -------------------------------------------------------------------------
output_path = results_folder / "topn_significance_results_corrected.xlsx"
final_results.to_excel(output_path, index=False)
print(f"\n✅ Results saved with Holm/FDR corrections:\n- {output_path}")

In [ ]:
# -------------------------------------------------------------------------
# --- CONFIGURATION
# -------------------------------------------------------------------------

alpha = 0.05

# -------------------------------------------------------------------------
# --- FUNCTION: Top-n significance testing
# -------------------------------------------------------------------------
def significance_tests_top_n(data, n_diagnoses, model_name, category):
    """
    Run Top-n significance tests vs. random baseline and return a results DataFrame.
    Includes CI, counts, and improvements.
    """
    N = len(data)
    results = []

    # Observed accuracies and counts
    accs = {k: data[f'Top_{k}_Accuracy'].mean() for k in [1, 2, 3]}
    counts = {k: data[f'Top_{k}_Accuracy'].sum() for k in [1, 2, 3]}
    expected = {k: k / n_diagnoses for k in [1, 2, 3]}

    # --- Top-k tests ---
    for k in [1, 2, 3]:
        count = counts[k]
        stat, pval = proportions_ztest(count, N, value=expected[k], alternative='larger')
        ci_low, ci_upp = proportion_confint(count, N, alpha=alpha, method='wilson')

        results.append({
            "Model": model_name,
            "Category": category,
            "Test_Type": f"Top-{k}",
            "Observed_Accuracy": accs[k],
            "Correct_Count": int(count),
            "N": N,
            "Expected_Random": expected[k],
            "z_value": stat,
            "p_value": pval,
            "CI_Lower": ci_low,
            "CI_Upper": ci_upp,
            "n_diagnoses": n_diagnoses
        })

    # --- Improvement tests (Top-1→Top-2, Top-2→Top-3) ---
    for (a, b) in [(1, 2), (2, 3)]:
        improvement = (data[f'Top_{b}_Accuracy'] & ~data[f'Top_{a}_Accuracy'])
        num_improved = improvement.sum()
        p_improvement = num_improved / N
        p_expected = (b - a) / n_diagnoses
        stat, pval = proportions_ztest(num_improved, N, value=p_expected, alternative='larger')
        ci_low, ci_upp = proportion_confint(num_improved, N, alpha=alpha, method='wilson')

        results.append({
            "Model": model_name,
            "Category": category,
            "Test_Type": f"Improvement Top{a}→Top{b}",
            "Observed_Accuracy": p_improvement,
            "Correct_Count": int(num_improved),
            "N": N,
            "Expected_Random": p_expected,
            "z_value": stat,
            "p_value": pval,
            "CI_Lower": ci_low,
            "CI_Upper": ci_upp,
            "n_diagnoses": n_diagnoses
        })

    return pd.DataFrame(results)


# -------------------------------------------------------------------------
# --- FUNCTION: multiple comparison correction
# -------------------------------------------------------------------------
def apply_multiple_corrections(df, alpha=0.05):
    """Apply Holm and FDR corrections to all p-values."""
    df = df.copy()
    valid = df['p_value'].notna()
    if valid.sum() == 0:
        return df

    df.loc[valid, 'p_holm'] = multipletests(df.loc[valid, 'p_value'], method='holm')[1]
    df.loc[valid, 'p_fdr'] = multipletests(df.loc[valid, 'p_value'], method='fdr_bh')[1]

    df['significant_holm'] = df['p_holm'] < alpha
    df['significant_fdr'] = df['p_fdr'] < alpha
    df['Correction_Significant'] = np.select(
        [df['significant_holm'], df['significant_fdr']],
        ['Holm', 'FDR'],
        default='None'
    )
    return df


# -------------------------------------------------------------------------
# --- LOOP OVER MODEL FILES
# -------------------------------------------------------------------------
n_choices = {"Mood": 13, "Stress": 8, "Anxiety": 16}
all_category_results = []
all_overall_results = []

files = sorted(results_folder.parent.glob("*/english/*detailed_results_corrected.csv"))

for file in files:
    model = file.parts[-3]
    df = pd.read_csv(file, index_col=['Vignette_ID', 'Category'])

    # --- Category-level ---
    for category in ["Anxiety", "Mood", "Stress"]:
        if category not in df.index.get_level_values("Category"):
            continue
        n_diagnoses = n_choices[category]
        df_cat = df.xs(category, level="Category")[['Top_1_Accuracy', 'Top_2_Accuracy', 'Top_3_Accuracy']]
        res_df = significance_tests_top_n(df_cat, n_diagnoses, model, category)
        all_category_results.append(res_df)

    # --- Overall (all vignettes) ---
    N_all = len(df)
    n_diagnoses_overall = int(np.mean(list(n_choices.values())))  # rough average baseline
    res_overall = significance_tests_top_n(
        df[['Top_1_Accuracy', 'Top_2_Accuracy', 'Top_3_Accuracy']],
        n_diagnoses_overall,
        model,
        category="Overall"
    )
    all_overall_results.append(res_overall)


# -------------------------------------------------------------------------
# --- COMBINE + CORRECT
# -------------------------------------------------------------------------
results_by_category = pd.concat(all_category_results, ignore_index=True)
results_overall = pd.concat(all_overall_results, ignore_index=True)

results_by_category = apply_multiple_corrections(results_by_category, alpha=alpha)
results_overall = apply_multiple_corrections(results_overall, alpha=alpha)


# -------------------------------------------------------------------------
# --- SAVE TO MULTI-SHEET EXCEL
# -------------------------------------------------------------------------
output_path = results_folder / "topn_significance_results_corrected.xlsx"

with pd.ExcelWriter(output_path) as writer:
    results_by_category.to_excel(writer, sheet_name="By_Category", index=False)
    results_overall.to_excel(writer, sheet_name="Overall", index=False)

print(f"\n✅ Results saved with Holm/FDR corrections:\n- {output_path}")


## Wilcoxon Tests 

In [ ]:
# -------------------------------------------------------------------------
# --- CONFIGURATION
# -------------------------------------------------------------------------

alpha = 0.05

# -------------------------------------------------------------------------
# --- FUNCTIONS
# -------------------------------------------------------------------------
def run_wilcoxon_test(data):
    """
    Runs a one-sided Wilcoxon signed-rank test comparing model vs. clinician performance.
    """
    data = data.dropna(subset=['Top_1_Accuracy', 'Mean Clinician Performance'])
    if data.empty:
        return np.nan, np.nan, 0

    stat, p_value = stats.wilcoxon(
        data['Top_1_Accuracy'],
        data['Mean Clinician Performance'],
        alternative='greater'
    )

    n = len(data)
    return stat, p_value, n


def apply_multiple_corrections(df, alpha=0.05):
    """
    Adds Holm and FDR-corrected p-values to the DataFrame, plus significance flags.
    """
    df = df.copy()
    if df.empty or 'p_value' not in df.columns:
        return df

    valid = df['p_value'].notna()
    if valid.sum() == 0:
        return df

    # Holm correction
    df.loc[valid, 'p_holm'] = multipletests(df.loc[valid, 'p_value'], method='holm')[1]
    # FDR (Benjamini–Hochberg)
    df.loc[valid, 'p_fdr'] = multipletests(df.loc[valid, 'p_value'], method='fdr_bh')[1]

    # Significance flags
    df['significant_holm'] = df['p_holm'] < alpha
    df['significant_fdr'] = df['p_fdr'] < alpha

    # Optional: compact summary column
    df['Correction_Significant'] = np.select(
        [df['significant_holm'], df['significant_fdr']],
        ['Holm', 'FDR'],
        default='None'
    )

    return df


# -------------------------------------------------------------------------
# --- LOAD CLINICIAN DATA
# -------------------------------------------------------------------------
clinicians = pd.read_excel(
    results_folder.parent.parent.joinpath('data', 'Clinician Rating Performance.xlsx'),
    sheet_name="Overall Perfromance Vignettes"
)

clinicians['Vignette_ID'] = clinicians['Vignette_ID'].astype(str)
clinicians = clinicians.set_index(["Category", "Vignette_ID"])
clinicians = clinicians / 100  # convert to proportion if originally in percentages

# -------------------------------------------------------------------------
# --- LOOP THROUGH MODEL FILES
# -------------------------------------------------------------------------
files = sorted(results_folder.parent.glob("*/english/*detailed_results_corrected.csv"))
all_results = []
all_overall_results = []

for file in files:
    model = file.parts[-3]
    df = pd.read_csv(file)
    df['Vignette_ID'] = [str(id.split(" ")[2]) for id in df['Vignette_ID']]

    # Merge with clinician data
    df_c = df.merge(clinicians, on=['Category', "Vignette_ID"], how="outer").set_index(["Category", "Vignette_ID"])
    df_c['Top_1_Accuracy'] = df_c['Top_1_Accuracy'].astype(float)

    # --- Per-category Wilcoxon ---
    for category in ["Anxiety", "Mood", "Stress"]:
        df_cat = df_c.xs(category, level="Category")[['Top_1_Accuracy', 'Mean Clinician Performance']].dropna()
        if df_cat.empty:
            continue

        stat, p_value, n = run_wilcoxon_test(df_cat)

        mean_model = df_cat['Top_1_Accuracy'].mean()
        mean_clin = df_cat['Mean Clinician Performance'].mean()
        mean_diff = mean_model - mean_clin

        all_results.append({
            "Model": model,
            "Category": category,
            "Test_Type": "Wilcoxon signed-rank (greater)",
            "W_statistic": stat,
            "p_value": p_value,
            "n": n,
            "Mean_Model_Performance": mean_model,
            "Mean_Clinician_Performance": mean_clin,
            "Mean_Difference": mean_diff
        })

    # --- Overall Wilcoxon (all vignettes) ---
    df_all = df_c[['Top_1_Accuracy', 'Mean Clinician Performance']].dropna()
    if not df_all.empty:
        stat, p_value, n = run_wilcoxon_test(df_all)

        mean_model = df_all['Top_1_Accuracy'].mean()
        mean_clin = df_all['Mean Clinician Performance'].mean()
        mean_diff = mean_model - mean_clin

        all_overall_results.append({
            "Model": model,
            "Category": "Overall",
            "Test_Type": "Wilcoxon signed-rank (greater)",
            "W_statistic": stat,
            "p_value": p_value,
            "n": n,
            "Mean_Model_Performance": mean_model,
            "Mean_Clinician_Performance": mean_clin,
            "Mean_Difference": mean_diff
        })

# -------------------------------------------------------------------------
# --- COMBINE RESULTS + APPLY CORRECTIONS
# -------------------------------------------------------------------------
results_by_category = pd.DataFrame(all_results)
results_overall = pd.DataFrame(all_overall_results)

results_by_category = apply_multiple_corrections(results_by_category, alpha=alpha)
results_overall = apply_multiple_corrections(results_overall, alpha=alpha)

# -------------------------------------------------------------------------
# --- SAVE TO MULTI-SHEET EXCEL
# -------------------------------------------------------------------------
output_path = results_folder / "Wilcoxon_significance_results_with_corrections.xlsx"
with pd.ExcelWriter(output_path) as writer:
    results_by_category.to_excel(writer, sheet_name="By_Category", index=False)
    results_overall.to_excel(writer, sheet_name="Overall", index=False)

print(f"\n✅ Results saved with Holm & FDR corrections:\n- {output_path}")


## Paired Wilcoxon signed-rank test with Hodges–Lehmann estimate & CIs

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

# -------------------------------------------------------------------------
# --- CONFIGURATION
# -------------------------------------------------------------------------
alpha = 0.05

# Practical/clinical margin (absolute difference in proportion units)
# δ = 0.05 means ±5 percentage points.
MARGIN = 0.10

# -------------------------------------------------------------------------
# --- EFFECT SIZE + CI FUNCTIONS (NEW)
# -------------------------------------------------------------------------
def hodges_lehmann_paired(diffs):
    """
    Hodges–Lehmann estimator for paired data (Wilcoxon signed-rank):
    median of Walsh averages (d_i + d_j)/2 for i <= j.
    """
    diffs = np.asarray(diffs, dtype=float)
    diffs = diffs[~np.isnan(diffs)]
    if diffs.size == 0:
        return np.nan

    walsh = (diffs[:, None] + diffs[None, :]) / 2.0
    walsh = walsh[np.triu_indices(diffs.size)]
    return float(np.median(walsh))


def _wilcoxon_pvalue_shift(diffs, delta, alternative="greater", zero_method="wilcox", method="auto"):
    """
    p-value for H0: median(diffs - delta) = 0 using Wilcoxon signed-rank test.
    """
    x = np.asarray(diffs, dtype=float) - float(delta)
    x = x[~np.isnan(x)]
    if x.size == 0:
        return np.nan

    try:
        _, p = stats.wilcoxon(
            x,
            alternative=alternative,
            zero_method=zero_method,
            method=method
        )
    except ValueError:
        # e.g., if all effective differences are zero after handling zeros
        return np.nan
    return float(p)


def wilcoxon_ci_by_inversion(
    diffs,
    conf_level=0.95,
    two_sided=True,
    zero_method="wilcox",
    method="auto",
    max_iter=80
):
    """
    Confidence interval for the Wilcoxon signed-rank location shift parameter
    obtained by inverting scipy.stats.wilcoxon.

    Returns:
      - (low, high) if two_sided=True
      - (low, np.inf) if two_sided=False  (one-sided lower CI)
    """
    diffs = np.asarray(diffs, dtype=float)
    diffs = diffs[~np.isnan(diffs)]
    if diffs.size == 0:
        return (np.nan, np.nan) if two_sided else (np.nan, np.inf)

    dmin, dmax = float(np.min(diffs)), float(np.max(diffs))
    span = dmax - dmin
    if span == 0:
        return (dmin, dmin) if two_sided else (dmin, np.inf)

    alpha_ci = 1.0 - conf_level

    if two_sided:
        target = alpha_ci / 2.0
        lower_target = target
        upper_target = target
    else:
        lower_target = alpha_ci
        upper_target = None

    # --- LOWER bound L:
    # L solves p_greater(diffs - L) = lower_target.
    # p_greater is monotone increasing in L.
    def p_lower(L):
        return _wilcoxon_pvalue_shift(
            diffs, L,
            alternative="greater",
            zero_method=zero_method,
            method=method
        )

    lo = dmin - 5 * span
    hi = dmax + 5 * span

    # attempt to bracket lower_target
    for _ in range(30):
        plo, phi = p_lower(lo), p_lower(hi)
        if np.isnan(plo) or np.isnan(phi):
            break
        if plo <= lower_target <= phi:
            break
        if plo > lower_target and phi > lower_target:
            lo -= 5 * span
            hi -= 5 * span
        elif plo < lower_target and phi < lower_target:
            lo += 5 * span
            hi += 5 * span
        else:
            break

    L = np.nan
    plo, phi = p_lower(lo), p_lower(hi)
    if not (np.isnan(plo) or np.isnan(phi)) and plo <= lower_target <= phi:
        a, b = lo, hi
        for _ in range(max_iter):
            m = (a + b) / 2.0
            pm = p_lower(m)
            if np.isnan(pm):
                break
            if pm < lower_target:
                a = m
            else:
                b = m
        L = float(b)

    if not two_sided:
        return (L, np.inf)

    # --- UPPER bound U:
    # U solves p_less(diffs - U) = upper_target.
    # p_less is monotone decreasing in U.
    def p_upper(U):
        return _wilcoxon_pvalue_shift(
            diffs, U,
            alternative="less",
            zero_method=zero_method,
            method=method
        )

    lo = dmin - 5 * span
    hi = dmax + 5 * span

    for _ in range(30):
        plo, phi = p_upper(lo), p_upper(hi)
        if np.isnan(plo) or np.isnan(phi):
            break
        # since decreasing: want phi <= target <= plo
        if phi <= upper_target <= plo:
            break
        if plo > upper_target and phi > upper_target:
            lo += 5 * span
            hi += 5 * span
        elif plo < upper_target and phi < upper_target:
            lo -= 5 * span
            hi -= 5 * span
        else:
            break

    U = np.nan
    plo, phi = p_upper(lo), p_upper(hi)
    if not (np.isnan(plo) or np.isnan(phi)) and phi <= upper_target <= plo:
        a, b = lo, hi
        for _ in range(max_iter):
            m = (a + b) / 2.0
            pm = p_upper(m)
            if np.isnan(pm):
                break
            if pm < upper_target:
                b = m
            else:
                a = m
        U = float(a)

    return (L, U)


def compute_hl_and_cis(diffs, zero_method="wilcox", method="auto"):
    """
    Returns HL estimate + CI90 (two-sided), CI95 (two-sided),
    and NI 95% one-sided lower bound.
    """
    hl = hodges_lehmann_paired(diffs)

    ci90_l, ci90_u = wilcoxon_ci_by_inversion(
        diffs, conf_level=0.90, two_sided=True,
        zero_method=zero_method, method=method
    )
    ci95_l, ci95_u = wilcoxon_ci_by_inversion(
        diffs, conf_level=0.95, two_sided=True,
        zero_method=zero_method, method=method
    )
    ni_lb95, _ = wilcoxon_ci_by_inversion(
        diffs, conf_level=0.95, two_sided=False,
        zero_method=zero_method, method=method
    )

    return hl, ci90_l, ci90_u, ci95_l, ci95_u, ni_lb95


# -------------------------------------------------------------------------
# --- ORIGINAL FUNCTIONS
# -------------------------------------------------------------------------
def run_wilcoxon_test(data):
    """
    Runs a one-sided Wilcoxon signed-rank test comparing model vs. clinician performance.
    H1: model > clinicians
    """
    data = data.dropna(subset=['Top_1_Accuracy', 'Mean Clinician Performance'])
    if data.empty:
        return np.nan, np.nan, 0

    stat, p_value = stats.wilcoxon(
        data['Top_1_Accuracy'],
        data['Mean Clinician Performance'],
        alternative='greater'
    )

    n = len(data)
    return stat, p_value, n


def apply_multiple_corrections(df, alpha=0.05):
    """
    Adds Holm and FDR-corrected p-values to the DataFrame, plus significance flags.
    """
    df = df.copy()
    if df.empty or 'p_value' not in df.columns:
        return df

    valid = df['p_value'].notna()
    if valid.sum() == 0:
        return df

    df.loc[valid, 'p_holm'] = multipletests(df.loc[valid, 'p_value'], method='holm')[1]
    df.loc[valid, 'p_fdr'] = multipletests(df.loc[valid, 'p_value'], method='fdr_bh')[1]

    df['significant_holm'] = df['p_holm'] < alpha
    df['significant_fdr'] = df['p_fdr'] < alpha

    df['Correction_Significant'] = np.select(
        [df['significant_holm'], df['significant_fdr']],
        ['Holm', 'FDR'],
        default='None'
    )
    return df


# -------------------------------------------------------------------------
# --- LOAD CLINICIAN DATA
# -------------------------------------------------------------------------
# NOTE: results_folder must already exist in your environment (as in your original script)
clinicians = pd.read_excel(
    results_folder.parent.parent.joinpath('data', 'Clinician Rating Performance.xlsx'),
    sheet_name="Overall Perfromance Vignettes"
)

clinicians['Vignette_ID'] = clinicians['Vignette_ID'].astype(str)
clinicians = clinicians.set_index(["Category", "Vignette_ID"])
clinicians = clinicians / 100  # convert to proportion if originally in percentages

# -------------------------------------------------------------------------
# --- LOOP THROUGH MODEL FILES
# -------------------------------------------------------------------------
files = sorted(results_folder.parent.glob("*/english/*detailed_results_corrected.csv"))
all_results = []
all_overall_results = []

for file in files:
    model = file.parts[-3]
    df = pd.read_csv(file)
    df['Vignette_ID'] = [str(id.split(" ")[2]) for id in df['Vignette_ID']]

    # Merge with clinician data
    df_c = df.merge(clinicians, on=['Category', "Vignette_ID"], how="outer").set_index(["Category", "Vignette_ID"])
    df_c['Top_1_Accuracy'] = df_c['Top_1_Accuracy'].astype(float)

    # --- Per-category Wilcoxon + HL/CI ---
    for category in ["Anxiety", "Mood", "Stress"]:
        df_cat = df_c.xs(category, level="Category")[['Top_1_Accuracy', 'Mean Clinician Performance']].dropna()
        if df_cat.empty:
            continue

        stat, p_value, n = run_wilcoxon_test(df_cat)

        mean_model = df_cat['Top_1_Accuracy'].mean()
        mean_clin = df_cat['Mean Clinician Performance'].mean()
        mean_diff = mean_model - mean_clin

        # NEW: HL effect size + CIs on paired differences (model - clinician)
        diffs = (df_cat['Top_1_Accuracy'] - df_cat['Mean Clinician Performance']).values
        hl, ci90_l, ci90_u, ci95_l, ci95_u, ni_lb95 = compute_hl_and_cis(diffs)

        # NEW: Decision flags using margin δ = 0.05
        equiv_90 = (ci90_l >= -MARGIN) and (ci90_u <= MARGIN)
        noninf_95 = (ni_lb95 > -MARGIN)  # AI not worse by more than MARGIN

        all_results.append({
            "Model": model,
            "Category": category,
            "Test_Type": "Wilcoxon signed-rank (greater)",
            "W_statistic": stat,
            "p_value": p_value,
            "n": n,
            "Mean_Model_Performance": mean_model,
            "Mean_Clinician_Performance": mean_clin,
            "Mean_Difference": mean_diff,

            # NEW outputs:
            "HL_Estimate_ModelMinusClinician": hl,
            "CI90_Lower": ci90_l,
            "CI90_Upper": ci90_u,
            "CI95_Lower": ci95_l,
            "CI95_Upper": ci95_u,
            "NI_OneSided95_Lower": ni_lb95,

            # NEW interpretation flags (δ = 0.05):
            "Equivalence_90CI_within_±0.05": equiv_90,
            "NonInferiority_95LB_above_-0.05": noninf_95,
        })

    # --- Overall Wilcoxon + HL/CI (all vignettes) ---
    df_all = df_c[['Top_1_Accuracy', 'Mean Clinician Performance']].dropna()
    if not df_all.empty:
        stat, p_value, n = run_wilcoxon_test(df_all)

        mean_model = df_all['Top_1_Accuracy'].mean()
        mean_clin = df_all['Mean Clinician Performance'].mean()
        mean_diff = mean_model - mean_clin

        diffs = (df_all['Top_1_Accuracy'] - df_all['Mean Clinician Performance']).values
        hl, ci90_l, ci90_u, ci95_l, ci95_u, ni_lb95 = compute_hl_and_cis(diffs)

        equiv_90 = (ci90_l >= -MARGIN) and (ci90_u <= MARGIN)
        noninf_95 = (ni_lb95 > -MARGIN)

        all_overall_results.append({
            "Model": model,
            "Category": "Overall",
            "Test_Type": "Wilcoxon signed-rank (greater)",
            "W_statistic": stat,
            "p_value": p_value,
            "n": n,
            "Mean_Model_Performance": mean_model,
            "Mean_Clinician_Performance": mean_clin,
            "Mean_Difference": mean_diff,

            # NEW outputs:
            "HL_Estimate_ModelMinusClinician": hl,
            "CI90_Lower": ci90_l,
            "CI90_Upper": ci90_u,
            "CI95_Lower": ci95_l,
            "CI95_Upper": ci95_u,
            "NI_OneSided95_Lower": ni_lb95,

            # NEW interpretation flags (δ = 0.05):
            "Equivalence_90CI_within_±0.05": equiv_90,
            "NonInferiority_95LB_above_-0.05": noninf_95,
        })

# -------------------------------------------------------------------------
# --- COMBINE RESULTS + APPLY CORRECTIONS (p-values only)
# -------------------------------------------------------------------------
results_by_category = pd.DataFrame(all_results)
results_overall = pd.DataFrame(all_overall_results)

results_by_category = apply_multiple_corrections(results_by_category, alpha=alpha)
results_overall = apply_multiple_corrections(results_overall, alpha=alpha)

# -------------------------------------------------------------------------
# --- SAVE TO MULTI-SHEET EXCEL
# -------------------------------------------------------------------------
output_path = results_folder / f"Wilcoxon_significance_results_with_corrections_and_HL_CI_{MARGIN}.xlsx"
with pd.ExcelWriter(output_path) as writer:
    results_by_category.to_excel(writer, sheet_name="By_Category", index=False)
    results_overall.to_excel(writer, sheet_name="Overall", index=False)

print(f"\n✅ Results saved (Wilcoxon + Holm/FDR + HL + CI + Equivalence/NI flags):\n- {output_path}")
print(f"Margin used for equivalence/non-inferiority: δ = {MARGIN:.2f} (absolute proportion units)")


The estimated differences are small-to-moderate, but confidence intervals are wide and cross the predefined equivalence and non-inferiority margins. Therefore, equivalence and non-inferiority cannot be concluded.

Failure to demonstrate equivalence or non-inferiority likely reflects limited statistical precision rather than evidence of clinically meaningful differences.

## Pair-wise McNemar tests

In [ ]:


# --- Load model correctness per vignette ---
files = sorted(results_folder.parent.glob("*/english/*detailed_results_corrected.csv"))
model_dfs_list = []

for file in files:
    model = file.parts[-3]
    df = pd.read_csv(file)
    df['Vignette_ID'] = [str(id.split(" ")[2]) for id in df['Vignette_ID']]
    df[model] = df['Top_1_Accuracy'].astype(int)
    df = df[["Category", "Vignette_ID", model]]
    df = df.set_index(["Category", "Vignette_ID"])
    model_dfs_list.append(df)

# Combine all models into one wide dataframe
model_dfs = pd.concat(model_dfs_list, axis=1)

categories = ["Anxiety", "Mood", "Stress"]
all_mcnemar_results = []

# --- Function to run pairwise McNemar tests ---
def run_mcnemar_tests(df_wide, category_label=None):
    results = []
    for model_a, model_b in combinations(df_wide.columns, 2):
        a, b = df_wide[model_a], df_wide[model_b]

        n_11 = ((a == 1) & (b == 1)).sum()
        n_10 = ((a == 1) & (b == 0)).sum()
        n_01 = ((a == 0) & (b == 1)).sum()
        n_00 = ((a == 0) & (b == 0)).sum()
        table = [[n_11, n_10],
                 [n_01, n_00]]

        if category_label:
            result = mcnemar(table, exact=True, correction=False)
            results.append({
                "Category": category_label,
                "Model_A": model_a,
                "Model_B": model_b,
                "b01": n_01,
                "b10": n_10,
                "p_value": result.pvalue,
                "n": n_01 + n_10
            })
        else:
            result = mcnemar(table, exact=False, correction=True)
            results.append({
                "Model_A": model_a,
                "Model_B": model_b,
                "b01": n_01,
                "b10": n_10,
                "chi2": result.statistic,
                "p_value": result.pvalue,
                "n": n_01 + n_10
            })
    return results

# --- Category-specific tests ---
for category in categories:
    df_wide = model_dfs.xs(category, level="Category", drop_level=False)
    cat_results = run_mcnemar_tests(df_wide, category)
    all_mcnemar_results.extend(cat_results)

# --- Overall tests across all vignettes (pooled) ---
df_overall = model_dfs.reset_index(drop=True)
overall_results = run_mcnemar_tests(df_overall, "Overall")
all_mcnemar_results.extend(overall_results)

# --- Build DataFrame and apply corrections ---
mcnemar_results = pd.DataFrame(all_mcnemar_results)

mcnemar_results['p_holm'] = multipletests(mcnemar_results['p_value'], method='holm')[1]
mcnemar_results['p_fdr']  = multipletests(mcnemar_results['p_value'], method='fdr_bh')[1]

alpha = 0.05
mcnemar_results['significant_holm'] = mcnemar_results['p_holm'] < alpha
mcnemar_results['significant_fdr']  = mcnemar_results['p_fdr'] < alpha

# --- Split into two DataFrames ---
by_category = mcnemar_results[mcnemar_results['Category'] != "Overall"]




#### All

# --- Overall tests across all vignettes (pooled) ---
df_overall = model_dfs.reset_index(drop=True)
overall_results = run_mcnemar_tests(df_overall)

# --- Build DataFrame and apply corrections ---
mcnemar_results_all = pd.DataFrame(overall_results)

mcnemar_results_all['p_holm'] = multipletests(mcnemar_results_all['p_value'], method='holm')[1]
mcnemar_results_all['p_fdr']  = multipletests(mcnemar_results_all['p_value'], method='fdr_bh')[1]

alpha = 0.05
mcnemar_results_all['significant_holm'] = mcnemar_results_all['p_holm'] < alpha
mcnemar_results_all['significant_fdr']  = mcnemar_results_all['p_fdr'] < alpha




# --- Save both to Excel (multi-sheet) ---
output_path = results_folder / "McNemar_pairwise_results.xlsx"
with pd.ExcelWriter(output_path) as writer:
    by_category.to_excel(writer, sheet_name="By_Category", index=False)
    mcnemar_results_all.to_excel(writer, sheet_name="Overall", index=False)

print(f"✅ Results saved with two sheets:\n- {output_path}")


## Cohen's Kappa

In [ ]:

files = sorted(results_folder.parent.glob("*/*/*detailed_results_corrected.csv"))
model_dfs_list = []
for file in files:
    model = file.parts[-3]
    df = pd.read_csv(file)
    df['Vignette_ID'] = [str(id.split(" ")[2]) for id in df['Vignette_ID']]
    df['Language'] = file.parts[-2]
    df['Model'] = model
    df['Top_1_Accuracy'] = df['Top_1_Accuracy'].astype(int)
    df = df[['Model',"Language","Category", "Vignette_ID","Top_1_Accuracy"]]
    df = df.set_index(['Model','Language',"Category", "Vignette_ID"])
    model_dfs_list.append(df)

# Combine all models into one wide dataframe
model_dfs = pd.concat(model_dfs_list)
categories = ["Anxiety", "Mood", "Stress"]
languages = df.index.get_level_values('Language').unique()
all_mcnemar_results = []

def run_language_mcnemar(df_long, model, category=None):
    if category: 
        # Slice by model + category without dropping other levels
        df_sub = df_long.xs(model, level="Model", drop_level=False)
        df_sub = df_sub.xs(category, level="Category", drop_level=False)

        results = []
        langs = df_sub.index.get_level_values('Language').unique()

        for lang_a, lang_b in combinations(langs, 2):
            # Binary scores for same vignettes in both languages
            a = df_sub.xs(lang_a, level="Language")['Top_1_Accuracy']
            b = df_sub.xs(lang_b, level="Language")['Top_1_Accuracy']

            # Align to common vignette IDs
            a, b = a.align(b, join='inner')

            # Compute contingency table
            n_11 = ((a == 1) & (b == 1)).sum()
            n_10 = ((a == 1) & (b == 0)).sum()
            n_01 = ((a == 0) & (b == 1)).sum()
            n_00 = ((a == 0) & (b == 0)).sum()
            table = [[n_11, n_10],
                    [n_01, n_00]]
            
            # If there are no discordant pairs, skip test
            if (n_01 + n_10) == 0:
                results.append({
                    "Model": model,
                    "Category": category,
                    "Language_A": lang_a,
                    "Language_B": lang_b,
                    "b01": n_01,
                    "b10": n_10,
                    "p_value": None,
                    "n": 0,
                    "Note": "No discordant pairs (identical performance)"
                })
                continue

            result = mcnemar(table, exact=True, correction=False)

            results.append({
                "Model": model,
                "Category": category,
                "Language_A": lang_a,
                "Language_B": lang_b,
                "b01": n_01,
                "b10": n_10,
                "p_value": result.pvalue,
                "n": n_01 + n_10
            })
        return results
    else:
        # Slice by model + category without dropping other levels
        df_sub = df_long.xs(model, level="Model", drop_level=False)

        results = []
        langs = df_sub.index.get_level_values('Language').unique()

        for lang_a, lang_b in combinations(langs, 2):
            # Binary scores for same vignettes in both languages
            a = df_sub.xs(lang_a, level="Language")['Top_1_Accuracy']
            b = df_sub.xs(lang_b, level="Language")['Top_1_Accuracy']

            # Align to common vignette IDs
            a, b = a.align(b, join='inner')

            # Compute contingency table
            n_11 = ((a == 1) & (b == 1)).sum()
            n_10 = ((a == 1) & (b == 0)).sum()
            n_01 = ((a == 0) & (b == 1)).sum()
            n_00 = ((a == 0) & (b == 0)).sum()
            table = [[n_11, n_10],
                    [n_01, n_00]]
            
            # If there are no discordant pairs, skip test
            if (n_01 + n_10) == 0:
                results.append({
                    "Model": model,
                    "Language_A": lang_a,
                    "Language_B": lang_b,
                    "b01": n_01,
                    "b10": n_10,
                    "chi2": None,
                    "p_value": None,
                    "n": 0,
                    "Note": "No discordant pairs (identical performance)"
                })
                continue

            result = mcnemar(table, exact=False, correction=True)

            results.append({
                "Model": model,
                "Language_A": lang_a,
                "Language_B": lang_b,
                "b01": n_01,
                "b10": n_10,
                "chi2": result.statistic,
                "p_value": result.pvalue,
                "n": n_01 + n_10
            })
        return results


# --- Run tests for each model across languages ---
all_mcnemar_results = []
models = model_dfs.index.get_level_values('Model').unique()

for model in models:
    for category in categories:
        results = run_language_mcnemar(model_dfs, model, category)
        all_mcnemar_results.extend(results)


# Optional pooled test (model across all languages, all categories)
for model in model_dfs.columns:
    df_overall = model_dfs.reset_index(drop=True)[model]

mcnemar_results = pd.DataFrame(all_mcnemar_results)
mcnemar_results['p_holm'] = multipletests(mcnemar_results['p_value'], method='holm')[1]
# mcnemar_results['p_fdr']  = multipletests(mcnemar_results['p_value'], method='fdr_bh')[1]

alpha = 0.05
mcnemar_results['significant_holm'] = mcnemar_results['p_holm'] < alpha
# mcnemar_results['significant_fdr']  = mcnemar_results['p_fdr'] < alpha


#### Overall 
all_mcnemar_results_all = []
models = model_dfs.index.get_level_values('Model').unique()

for model in models:
    results = run_language_mcnemar(model_dfs, model)
    all_mcnemar_results_all.extend(results)

# Optional pooled test (model across all languages, all categories)
for model in model_dfs.columns:
    df_overall = model_dfs.reset_index(drop=True)[model]

mcnemar_results_all = pd.DataFrame(all_mcnemar_results_all)
mcnemar_results_all['p_holm'] = multipletests(mcnemar_results_all['p_value'], method='holm')[1]

# --- Split into two DataFrames ---
by_category = mcnemar_results[mcnemar_results['Category'] != "Overall"]
overall = mcnemar_results_all #[mcnemar_results_all['Category'] == "Overall"]

# --- Save both to Excel (multi-sheet) ---
output_path = results_folder / "McNemar_pairwise_results_multilingual.xlsx"
with pd.ExcelWriter(output_path) as writer:
    by_category.to_excel(writer, sheet_name="By_Category", index=False)
    overall.to_excel(writer, sheet_name="Overall", index=False)

print(f"✅ Results saved with two sheets:\n- {output_path}")
